In [1]:
import numpy as np
import pandas as pd
import json
import torch
import torch.nn.functional as F
from scipy.signal.windows import gaussian
from scipy.signal import find_peaks
import h5py
import re
import plotly.graph_objects as go
from typing import List, Dict, Tuple, Optional
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Konfigurasi parameter simulasi
SIMULATION_CONFIG = {
    "resolution": 24480,
    "wl_range": (200, 900),
    "sigma": 0.1,
    "target_max_intensity": 0.8,
    "convolution_sigma": 0.1,
}

# Konstanta fisika
PHYSICAL_CONSTANTS = {
    "k_B": 8.617333262145e-5,  # eV/K
    "m_e": 9.1093837e-31,      # kg
    "h": 4.135667696e-15,      # eV·s
}

# Elemen target
BASE_ELEMENTS = ['Al', 'Ca', 'Fe', 'Si', 'Mg', 'Ti', 'Cr', 'Mn', 'Ni', 'Cu', 'Li','Ca', 'Mg', 'Na', 'K', 'O', 'H', 'N', 'Sr', 'Si', "C", 'Rb',"Co", "Pb"]

def calculate_lte_electron_density(temp: float, delta_E: float) -> float:
    return 1.6e12 * (temp ** 0.5) * (delta_E ** 3)

class DataFetcher:
    def __init__(self, hdf_path: str):
        self.hdf_path = hdf_path
        self.delta_E_max = {}

    def get_nist_data(self, element: str, sp_num: int) -> Tuple[List[List], float]:
        try:
            with pd.HDFStore(self.hdf_path, mode='r') as store:
                df = store.get('nist_spectroscopy_data')
                filtered_df = df[(df['element'] == element) & (df['sp_num'] == sp_num)]
                required_columns = ['ritz_wl_air(nm)', 'Aki(s^-1)', 'Ek(eV)', 'Ei(eV)', 'g_i', 'g_k']
                if filtered_df.empty or not all(col in df.columns for col in required_columns):
                    return [], 0.0
                filtered_df = filtered_df.dropna(subset=required_columns)

                filtered_df['ritz_wl_air(nm)'] = pd.to_numeric(filtered_df['ritz_wl_air(nm)'], errors='coerce')
                for col in ['Ek(eV)', 'Ei(eV)']:
                    filtered_df[col] = filtered_df[col].apply(
                        lambda x: float(re.sub(r'[^\d.-]', '', str(x))) if re.sub(r'[^\d.-]', '', str(x)) else None
                    )
                filtered_df = filtered_df.dropna(subset=['ritz_wl_air(nm)', 'Ek(eV)', 'Ei(eV)'])

                filtered_df = filtered_df[
                    (filtered_df['ritz_wl_air(nm)'] >= SIMULATION_CONFIG["wl_range"][0]) &
                    (filtered_df['ritz_wl_air(nm)'] <= SIMULATION_CONFIG["wl_range"][1])
                ]
                filtered_df['delta_E'] = abs(filtered_df['Ek(eV)'] - filtered_df['Ei(eV)'])
                if not filtered_df.empty:
                    filtered_df = filtered_df.sort_values(by='Aki(s^-1)', ascending=False)
                    delta_E_max = filtered_df['delta_E'].max()
                    if pd.isna(delta_E_max): delta_E_max = 0.0
                else:
                    delta_E_max = 0.0
                self.delta_E_max[f"{element}_{sp_num}"] = delta_E_max
                return filtered_df[required_columns + ['Acc']].values.tolist(), delta_E_max
        except Exception as e:
            print(f"Error fetching NIST data for {element}_{sp_num}: {str(e)}")
            return [], 0.0

class SpectrumSimulator:
    def __init__(
        self,
        nist_data: List[List],
        element: str,
        ion: int,
        temperature: float,
        ionization_energy: float,
        config: Dict = SIMULATION_CONFIG
    ):
        self.nist_data = nist_data
        self.element = element
        self.ion = ion
        self.temperature = temperature
        self.ionization_energy = ionization_energy
        self.resolution = config["resolution"]
        self.wl_range = config["wl_range"]
        self.sigma = config["sigma"]
        self.wavelengths = np.linspace(self.wl_range[0], self.wl_range[1], self.resolution, dtype=np.float32)
        self.gaussian_cache = {}
        self.element_label = f"{element} {'I' if self.ion == 1 else 'II'}"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def partition_function(self, energy_levels: List[float], degeneracies: List[float]) -> float:
        k_B = PHYSICAL_CONSTANTS["k_B"]
        return sum(g * np.exp(-E / (k_B * self.temperature)) for g, E in zip(degeneracies, energy_levels) if E is not None) or 1.0

    def calculate_intensity(self, energy: float, degeneracy: float, einstein_coeff: float, Z: float) -> float:
        k_B = PHYSICAL_CONSTANTS["k_B"]
        return (degeneracy * einstein_coeff * np.exp(-energy / (k_B * self.temperature))) / Z

    def gaussian_profile(self, center: float) -> np.ndarray:
        if center not in self.gaussian_cache:
            x_tensor = torch.tensor(self.wavelengths, device=self.device, dtype=torch.float32)
            center_tensor = torch.tensor(center, device=self.device, dtype=torch.float32)
            sigma_tensor = torch.tensor(self.sigma, device=self.device, dtype=torch.float32)
            gaussian_val = torch.exp(-0.5 * ((x_tensor - center_tensor) / sigma_tensor) ** 2) / (sigma_tensor * torch.sqrt(torch.tensor(2 * np.pi, device=self.device)))
            self.gaussian_cache[center] = gaussian_val.cpu().numpy().astype(np.float32)
        return self.gaussian_cache[center]

    def simulate(self, atom_percentage: float = 1.0) -> Tuple[np.ndarray, np.ndarray, List[Tuple[float, float, str]]]:
        spectrum = torch.zeros(self.resolution, device=self.device, dtype=torch.float32)
        levels = {}
        prominent_lines = []

        for data in self.nist_data:
            try:
                _, _, Ek, Ei, gi, gk, _ = data
                if all(v is not None for v in [Ek, Ei, gi, gk]):
                    levels.setdefault(float(Ei), float(gi))
                    levels.setdefault(float(Ek), float(gk))
            except (ValueError, TypeError):
                continue

        if not levels: return self.wavelengths, np.zeros(self.resolution, dtype=np.float32), []

        energy_levels, degeneracies = list(levels.keys()), list(levels.values())
        Z = self.partition_function(energy_levels, degeneracies)

        for data in self.nist_data:
            try:
                wl, Aki, Ek, _, _, gk, _ = data
                if all(v is not None for v in [wl, Aki, Ek, gk]):
                    wl, Aki, Ek, gk = float(wl), float(Aki), float(Ek), float(gk)
                    intensity = self.calculate_intensity(Ek, gk, Aki, Z)
                    prominent_lines.append((wl, intensity * atom_percentage, self.element_label))

                    idx = np.searchsorted(self.wavelengths, wl)
                    if 0 <= idx < self.resolution:
                        gaussian_contrib = torch.tensor(
                            intensity * atom_percentage * self.gaussian_profile(wl),
                            device=self.device, dtype=torch.float32
                        )
                        start_idx = max(0, idx - len(gaussian_contrib) // 2)
                        end_idx = min(self.resolution, start_idx + len(gaussian_contrib))
                        if start_idx < end_idx:
                            spectrum[start_idx:end_idx] += gaussian_contrib[:end_idx - start_idx]
            except (ValueError, TypeError):
                continue

        prominent_lines.sort(key=lambda x: x[1], reverse=True)
        return self.wavelengths, spectrum.cpu().numpy(), prominent_lines[:20]

class MixedSpectrumSimulator:
    def __init__(self, simulators: List[SpectrumSimulator], electron_density: float, delta_E_max: Dict[str, float], config: Dict = SIMULATION_CONFIG):
        self.simulators = simulators
        self.resolution = config["resolution"]
        self.wl_range = config["wl_range"]
        self.convolution_sigma = config["convolution_sigma"]
        self.electron_density = electron_density
        self.delta_E_max = delta_E_max
        self.wavelengths = np.linspace(self.wl_range[0], self.wl_range[1], self.resolution, dtype=np.float32)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def normalize_intensity(self, intensity: np.ndarray, target_max: float) -> np.ndarray:
        intensity_tensor = torch.tensor(intensity, device=self.device, dtype=torch.float32)
        max_intensity = torch.max(torch.abs(intensity_tensor))
        return intensity if max_intensity == 0 else (intensity_tensor / max_intensity * target_max).cpu().numpy()

    def convolve_spectrum(self, spectrum: np.ndarray, sigma_nm: float) -> np.ndarray:
        spectrum_tensor = torch.tensor(spectrum, device=self.device, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        wavelength_step = (self.wavelengths[-1] - self.wavelengths[0]) / (len(self.wavelengths) - 1)
        sigma_points = sigma_nm / wavelength_step
        kernel_size = int(6 * sigma_points) | 1
        kernel_np = gaussian(kernel_size, sigma_points)
        kernel = torch.tensor(kernel_np / kernel_np.sum(), device=self.device, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        convolved = F.conv1d(spectrum_tensor, kernel, padding=kernel_size//2).squeeze().cpu().numpy()
        return convolved.astype(np.float32)

    def saha_ratio(self, ion_energy: float, temp: float) -> float:
        k_B_eV = PHYSICAL_CONSTANTS["k_B"]
        m_e_kg = PHYSICAL_CONSTANTS["m_e"]
        h_eVs = PHYSICAL_CONSTANTS["h"]
        eV_to_J = 1.60217662e-19
        k_B_J = k_B_eV * eV_to_J
        h_J = h_eVs * eV_to_J
        kT_J = k_B_J * temp
        thermal_term = (2 * np.pi * m_e_kg * kT_J / (h_J**2)) ** 1.5
        n_e_m3 = self.electron_density * 1e6
        pre_exponential_factor = (2.0 / 1.0) * thermal_term / n_e_m3
        exp_term = np.exp(-ion_energy / (k_B_eV * temp))
        return pre_exponential_factor * exp_term

    def validate_lte(self, temperature: float, selected_elements: List[Tuple[str, float]]) -> Tuple[float, float]:
        delta_E_values = [v for k, v in self.delta_E_max.items() if any(k.startswith(elem) for elem, _ in selected_elements) and v > 0]
        delta_E_max = max(delta_E_values) if delta_E_values else 4.0
        n_e_min = calculate_lte_electron_density(temperature, delta_E_max)
        return delta_E_max, n_e_min

    def generate_spectrum(self, selected_elements: List[Tuple[str, float]], temperature: float) -> Tuple[np.ndarray, np.ndarray, Dict, List[Tuple[float, float, str]]]:
        mixed_spectrum = np.zeros(self.resolution, dtype=np.float32)
        all_prominent_lines = []
        atom_percentages_dict = {}

        delta_E_max, n_e_min = self.validate_lte(temperature, selected_elements)
        if self.electron_density < n_e_min:
            print(f"Warning: Electron density {self.electron_density:.1e} cm^-3 is below LTE requirement "
                  f"({n_e_min:.1e} cm^-3) for T={temperature:.0f} K, ΔE_max={delta_E_max:.2f} eV")
            self.electron_density = n_e_min
            print(f"Using n_e = {self.electron_density:.1e} cm^-3 to satisfy LTE")

        total_percentage = sum(p for _, p in selected_elements)
        if abs(total_percentage - 100.0) > 1e-6:
            raise ValueError(f"Total percentage ({total_percentage:.1f}%) must be exactly 100%")

        for base_elem, percentage in selected_elements:
            ion_energy = ionization_energies.get(f"{base_elem} I", 0.0)
            if ion_energy == 0.0:
                print(f"Warning: No ionization energy for {base_elem} I, skipping.")
                atom_percentages_dict[f"{base_elem}_1"] = 0
                atom_percentages_dict[f"{base_elem}_2"] = 0
                continue
            
            saha_val = self.saha_ratio(ion_energy, temperature)
            neutral_fraction = 1 / (1 + saha_val)
            ion_fraction = 1 - neutral_fraction
            atom_percentages_dict[f"{base_elem}_1"] = percentage * neutral_fraction / 100.0
            atom_percentages_dict[f"{base_elem}_2"] = percentage * ion_fraction / 100.0

        for simulator in self.simulators:
            elem_label_key = f"{simulator.element}_{simulator.ion}"
            if elem_label_key in atom_percentages_dict:
                simulator.temperature = temperature
                _, spectrum, prominent_lines = simulator.simulate(atom_percentages_dict[elem_label_key])
                mixed_spectrum += spectrum
                all_prominent_lines.extend(prominent_lines)

        if np.max(mixed_spectrum) == 0:
            print(f"Warning: No spectrum generated for temperature {temperature}")
            return self.wavelengths, np.zeros(self.resolution, dtype=np.float32), {}, []

        convolved_spectrum = self.convolve_spectrum(mixed_spectrum, self.convolution_sigma)
        normalized_spectrum = self.normalize_intensity(convolved_spectrum, SIMULATION_CONFIG["target_max_intensity"])

        final_composition = {k.replace('_', ' '): v * 100 for k, v in atom_percentages_dict.items()}
        final_composition['temperature'] = temperature
        final_composition['electron_density'] = self.electron_density
        final_composition['delta_E_max'] = delta_E_max
        final_composition['n_e_min'] = n_e_min

        return self.wavelengths, normalized_spectrum, final_composition, all_prominent_lines

def plot_spectrum_plotly(wavelengths, spectrum, temperature, electron_density, atom_percentages, all_prominent_lines, show_labels=True):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=wavelengths, y=spectrum, mode='lines', name='Simulated Spectrum', line=dict(color='navy', width=1.5)))

    if np.max(spectrum) > 0 and show_labels:
      # Anda bisa mengatur 'height' untuk mengontrol seberapa banyak puncak yang akan diberi label.
      # Nilai yang lebih kecil akan menampilkan lebih banyak label.
      height_threshold = 0.001 * np.max(spectrum)
      peak_indices, _ = find_peaks(spectrum, height=height_threshold)
      peak_wavelengths = wavelengths[peak_indices]
      peak_intensities = spectrum[peak_indices]

      annotations = []
      all_prominent_lines.sort(key=lambda x: x[1], reverse=True)

      for wl, intensity in zip(peak_wavelengths, peak_intensities):
          if all_prominent_lines:
              closest_line = min(all_prominent_lines, key=lambda line: abs(line[0] - wl))
              label = f"{closest_line[2]}<br>{closest_line[0]:.2f} nm"
              annotations.append(go.layout.Annotation(
                  x=wl, y=intensity, text=label, showarrow=True, arrowhead=2, arrowsize=1,
                  arrowwidth=1, ax=0, ay=-40, font=dict(size=9, color="#ffffff"),
                  align="center", bordercolor="#555", borderwidth=1, borderpad=2,
                  bgcolor="#ff7f0e", opacity=0.8))
      fig.update_layout(annotations=annotations)

    comp_items = [f'<b>{elem}</b>: {perc:.2f}%' for elem, perc in atom_percentages.items() if elem not in ['temperature', 'electron_density', 'delta_E_max', 'n_e_min']]
    param_items = [
        f'<b>&Delta;E_max</b>: {atom_percentages.get("delta_E_max", 0):.2f} eV',
        f'<b>n_e_min</b>: {atom_percentages.get("n_e_min", 0):.1e} cm<sup>-3</sup>']
    comp_text = '<b>Composition:</b><br>' + '<br>'.join(comp_items) + '<br><br>' + '<br>'.join(param_items)

    fig.update_layout(
        title=f'<b>Simulated Spectrum</b><br>T = {temperature:.0f} K, n_e = {electron_density:.1e} cm<sup>-3</sup>',
        xaxis_title='Wavelength (nm)', yaxis_title='Normalized Intensity (a.u.)',
        xaxis=dict(range=[wavelengths[0], wavelengths[-1]], showgrid=True, gridwidth=1, gridcolor='LightGray', minor=dict(showgrid=True, gridcolor='lightgrey', griddash='dot')),
        yaxis=dict(range=[-0.05, max(spectrum.max() * 1.15, 0.1)], showgrid=True, gridwidth=1, gridcolor='LightGray', minor=dict(showgrid=True, gridcolor='lightgrey', griddash='dot')),
        legend=dict(x=0.01, y=0.99, bordercolor='Black', borderwidth=1),
        font=dict(family="Arial, sans-serif", size=12, color="black"),
        plot_bgcolor='white', paper_bgcolor='white',
        annotations=fig.layout.annotations + (go.layout.Annotation(
            x=0.98, y=0.98, xref='paper', yref='paper', text=comp_text, showarrow=False,
            align='left', valign='top', font=dict(size=10), bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='black', borderwidth=1),)
    )
    fig.show()

def create_composition_form():
    element_options = [(elem, elem) for elem in BASE_ELEMENTS]
    composition_widgets = []
    total_percentage_label = widgets.Label(value="Total Percentage: 0.0%")

    def add_element_row(_=None):
        element_dropdown = widgets.Dropdown(options=element_options, description="Element:", layout={'width': '300px'})
        percentage_input = widgets.FloatText(value=0.0, description="Percentage (%):", layout={'width': '200px'})
        remove_button = widgets.Button(description="Remove", button_style='danger')

        def update_total(_=None):
            total = sum(w[1].value for w in composition_widgets)
            total_percentage_label.value = f"Total Percentage: {total:.1f}%"

        def remove_row(b):
            row_to_remove = next((row for row in composition_widgets if row[2] == b), None)
            if row_to_remove:
                composition_widgets.remove(row_to_remove)
                rows_vbox.children = [widgets.HBox(list(row)) for row in composition_widgets]
                update_total()

        percentage_input.observe(update_total, names='value')
        remove_button.on_click(remove_row)
        composition_widgets.append((element_dropdown, percentage_input, remove_button))
        rows_vbox.children = [widgets.HBox(list(row)) for row in composition_widgets]
        update_total()

    add_button = widgets.Button(description="Add Element", button_style='success')
    add_button.on_click(add_element_row)

    rows_vbox = widgets.VBox([])
    control_hbox = widgets.HBox([add_button, total_percentage_label])
    display(widgets.VBox([rows_vbox, control_hbox]))
    add_element_row()

    submit_button = widgets.Button(description="Generate Spectrum", button_style='primary')
    
    # Create checkbox for show labels toggle
    show_labels_checkbox = widgets.Checkbox(
        value=True,
        description='Show Peak Labels',
        style={'description_width': 'initial'}
    )
    
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output()
            selected_elements = [(w[0].value, w[1].value) for w in composition_widgets if w[1].value > 0]
            if not selected_elements:
                print("Error: No elements with percentage > 0 selected")
                return
            
            total_percentage = sum(p for _, p in selected_elements)
            if abs(total_percentage - 100.0) > 1e-6:
                print(f"Error: Total percentage ({total_percentage:.1f}%) must be exactly 100%")
                return

            run_simulation(selected_elements, temperature_input.value, electron_density_input.value, show_labels_checkbox.value)

    submit_button.on_click(on_submit)
    temperature_input = widgets.FloatText(value=10000, description="Temperature (K):", layout={'width': '300px'})

    ne_options_values = [10**exp for exp in np.arange(15, 18.1, 0.1)]
    ne_options_labels = [f"{val:.1e}" for val in ne_options_values]
    ne_options = list(zip(ne_options_labels, ne_options_values))
    desired_default = 1e17
    actual_default = min(ne_options_values, key=lambda x: abs(x - desired_default))
    electron_density_input = widgets.Dropdown(options=ne_options, value=actual_default, description="Electron Density (cm^-3):", layout={'width': '400px'})

    display(temperature_input, electron_density_input, show_labels_checkbox, submit_button, output)

def run_simulation(selected_elements, temperature, electron_density, show_labels=True):
    data_dir = "data"
    # Perbaikan nama file agar konsisten
    nist_path = "Nist_data(1).h5"
    atomic_data_path = "atomic_data1.h5"

    if not os.path.exists(nist_path) or not os.path.exists(atomic_data_path):
        print(f"Error: Pastikan file '{os.path.basename(nist_path)}' dan '{os.path.basename(atomic_data_path)}' ada di dalam folder '{data_dir}/'")
        return

    global ionization_energies
    ionization_energies = {}
    with h5py.File(atomic_data_path, 'r') as f:
        dset = f['elements']
        columns = dset.attrs['columns']
        data = []
        for item in dset[:]:
            row = [
                item[0],
                item[1].decode('utf-8'), item[2].decode('utf-8'),
                item[3].decode('utf-8'), item[4].decode('utf-8'),
                item[5], item[6].decode('utf-8')
            ]
            data.append(row)
        df_ionization = pd.DataFrame(data, columns=columns)
        for _, row in df_ionization.iterrows():
            ionization_energies[row["Sp. Name"]] = float(row["Ionization Energy (eV)"])

    fetcher = DataFetcher(nist_path)
    simulators = []
    delta_E_max_dict = {}
    unique_elements = {elem for elem, _ in selected_elements}

    for elem in unique_elements:
        for ion in [1, 2]:
            nist_data, delta_E = fetcher.get_nist_data(elem, ion)
            if not nist_data:
                continue
            delta_E_max_dict[f"{elem}_{ion}"] = delta_E
            ion_energy = ionization_energies.get(f"{elem} {'I' if ion == 1 else 'II'}", 0.0)
            simulator = SpectrumSimulator(nist_data, elem, ion, temperature, ion_energy)
            simulators.append(simulator)

    if not simulators:
        print("No valid simulators created based on the selected elements. Exiting.")
        return

    mixed_simulator = MixedSpectrumSimulator(simulators, electron_density, delta_E_max_dict)
    wavelengths, spectrum, atom_percentages, all_prominent_lines = mixed_simulator.generate_spectrum(selected_elements, temperature)

    if atom_percentages:
        print(f"Spectrum generated with:")
        print(f"Temperature: {temperature:.0f} K")
        print(f"Electron Density: {atom_percentages['electron_density']:.1e} cm^-3")
        print(f"ΔE_max: {atom_percentages['delta_E_max']:.2f} eV")
        print(f"n_e_min: {atom_percentages['n_e_min']:.1e} cm^-3")
        print("Atomic Composition (after Saha adjustment):")
        for elem, percentage in atom_percentages.items():
            if elem not in ['temperature', 'electron_density', 'delta_E_max', 'n_e_min']:
                print(f"  {elem}: {percentage:.2f}%")

        plot_spectrum_plotly(wavelengths, spectrum, temperature, atom_percentages['electron_density'], atom_percentages, all_prominent_lines, show_labels)

def main():
    if 'ipykernel' in __import__('sys').modules:
        print("Menjalankan dalam mode interaktif (Jupyter/IPython). Menampilkan UI.")
        create_composition_form()
    else:
        print("Menjalankan dalam mode skrip non-interaktif.")
        selected = [("Si", 25.0), ("Al", 25.0), ("Fe", 50.0)]
        temp = 12000
        ne = 1e17
        print(f"\nMemulai simulasi dengan parameter yang telah ditentukan:")
        print(f"Komposisi: {selected}")
        print(f"Suhu: {temp} K")
        print(f"Densitas Elektron: {ne:.1e} cm^-3\n")
        run_simulation(selected, temp, ne, show_labels=False)

if __name__ == "__main__":
    main()

Menjalankan dalam mode interaktif (Jupyter/IPython). Menampilkan UI.


FloatText(value=10000.0, description='Temperature (K):', layout=Layout(width='300px'))

Dropdown(description='Electron Density (cm^-3):', index=20, layout=Layout(width='400px'), options=(('1.0e+15',…

Checkbox(value=True, description='Show Peak Labels', style=CheckboxStyle(description_width='initial'))

Button(button_style='primary', description='Generate Spectrum', style=ButtonStyle())

Output()